# Phase 2 Kaggle Execution Notebook

This notebook is the full Kaggle execution entry point for the frozen five-language XNLI pilot.
It installs the project, validates the runtime, checks model access, prepares the fixed protocol manifest/data, and then exits cleanly after setup validation.

This notebook installs the Phase 2 setup package, validates Kaggle access, and creates the fixed XNLI data manifest. It does not start training. Runnable reference-baseline jobs are in the separate B0 and B1 Kaggle notebooks; adapters for B2-B8 are still being implemented or feasibility-checked.


In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/asymmetric-interference-continual-learning')
if not repo_root.exists():
    !git clone --depth 1 https://github.com/Mani-S-49/asymmetric-interference-continual-learning.git /kaggle/working/asymmetric-interference-continual-learning

%cd /kaggle/working/asymmetric-interference-continual-learning
!pip install -q -r phase2/requirements-phase2.txt

print('Repo ready at:', repo_root)


In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/asymmetric-interference-continual-learning')
sys.path.insert(0, str(repo_root))

def get_secret(name: str):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for secret_name, env_name in [('HF_TOKEN', 'HF_TOKEN'), ('WANDB_API_KEY', 'WANDB_API_KEY')]:
    value = get_secret(secret_name)
    if value:
        os.environ[env_name] = value

print('HF_TOKEN present:', bool(os.environ.get('HF_TOKEN')) )
print('WANDB_API_KEY present:', bool(os.environ.get('WANDB_API_KEY')) )
print('Repo root:', repo_root)
print('Python:', sys.version.split()[0])


In [ ]:
import json
import os
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/asymmetric-interference-continual-learning')
sys.path.insert(0, str(repo_root))

from phase2.preflight import collect_preflight
from phase2.protocol import build_manifest, load_protocol

protocol_path = repo_root / 'phase2' / 'configs' / 'pilot_protocol.json'
protocol = load_protocol(protocol_path)
manifest = build_manifest(protocol)

report = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(report, indent=2, sort_keys=True))
print('Protocol:', protocol['protocol_id'])
print('Languages:', protocol['dataset']['languages'])
print('Orders:', len(protocol['orders']))
print('Scheduled runs:', manifest['expected_run_count'])
print('Replay buffer capacity:', protocol['replay']['buffer_capacity_examples'])


In [ ]:
import json
import sys
from pathlib import Path

repo_root = Path('/kaggle/working/asymmetric-interference-continual-learning')
sys.path.insert(0, str(repo_root))

from phase2.prepare_data import prepare_xnli

protocol_path = repo_root / 'phase2' / 'configs' / 'pilot_protocol.json'
output_dir = repo_root / 'phase2_artifacts'
manifest = prepare_xnli(protocol_path, output_dir)
print(json.dumps({
    'protocol_id': manifest['protocol_id'],
    'manifest_path': str(output_dir / 'protocol_manifest.json'),
    'expected_run_count': manifest['expected_run_count'],
    'artifact_count': sum(len(splits) for splits in manifest['data_artifacts'].values()),
}, indent=2, sort_keys=True))

print('Prepared data directories:')
for language, splits in manifest['data_artifacts'].items():
    print(language, sorted(splits))


## Notes

The setup phase is complete and the fixed protocol/data manifest has been generated.
This is the setup and manifest-generation notebook only. Run B0 and B1 from their dedicated Kaggle notebooks after saving the prepared data artifacts; B2-B8 are not yet ready for full training runs.
